# Importing

In [ ]:
import sys
sys.path.append("home/565/pv3484/aus_substation_electricity")

%cd aus_substation_electricity/
!pwd

In [ ]:
%run /home/565/pv3484/aus_substation_electricity/import_substation.py

# Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
import os
from datetime import date, timedelta
from dateutil.easter import easter

In [ ]:
def second_monday_of_june(y):
    """Return the date of the second Monday in June for year y (Monarch's Birthday)."""
    june = pd.date_range(start=f"{y}-06-01", end=f"{y}-06-30", freq="D")
    mondays = june[june.weekday == 0]
    return mondays[1]


HOLIDAYS_VIC = {
    "New Year's Day":      lambda y: pd.Timestamp(f"{y}-01-01"),
    "Australia Day":       lambda y: pd.Timestamp(f"{y}-01-26"),
    "Good Friday":         lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=2),
    "Easter Saturday":     lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=1),
    "Easter Sunday":       lambda y: pd.Timestamp(easter(y)),
    "Easter Monday":       lambda y: pd.Timestamp(easter(y)) + pd.Timedelta(days=1),
    "ANZAC Day":           lambda y: pd.Timestamp(f"{y}-04-25"),
    "Monarch's Birthday":  lambda y: second_monday_of_june(y),
    "Christmas Day":       lambda y: pd.Timestamp(f"{y}-12-25"),
    "Boxing Day":          lambda y: pd.Timestamp(f"{y}-12-26"),
}

holiday_order = list(HOLIDAYS_VIC.keys())

# Maps individual holiday names to their group label
HOLIDAY_GROUPS = {
    "Good Friday":     "Easter Long Weekend",
    "Easter Saturday": "Easter Long Weekend",
    "Easter Sunday":   "Easter Long Weekend",
    "Easter Monday":   "Easter Long Weekend",
    "Christmas Day":   "Christmas and Boxing Day",
    "Boxing Day":      "Christmas and Boxing Day",
}

# CI plot function
- red line = mean 24 hour anomaly profile on holiday
- salmon band = 95% confidence interval around mean
- grey = ±1SD of ALL days in ±30 day window
- lower panel = per hour bar chart from one-sample t-tsts (red bar = significant)
- printed stats = n^2 effect size and overall ANOVA f/p

In [ ]:
def compare_holiday_ci(
    demand,
    info,
    station,
    years,
    holiday_func,
    holiday_name,
    alpha=0.05,
):
    """
    Holiday anomaly plot with:
    - Grey band  : min/max envelope of ALL non-holiday days in ±30-day window
    - Salmon band: min/max envelope across individual holiday-year curves
    - Red line   : mean holiday anomaly profile
    - Lower panel: per-hour t-test p-values + eta² / ANOVA stats

    Using min/max envelopes means the band always hugs what was actually
    observed, regardless of how many years are in the sample.

    Parameters
    ----------
    demand       : pd.DataFrame  — raw demand data (datetime index)
    info         : pd.DataFrame  — station metadata (index = station codes, column 'Name')
    station      : str           — station code, e.g. 'BLAKE'
    years        : tuple         — (start_year, end_year) inclusive, e.g. (2004, 2018)
    holiday_func : callable      — year -> pd.Timestamp
    holiday_name : str           — display name
    alpha        : float         — significance level for t-test threshold (default 0.05)

    Returns
    -------
    fig : matplotlib Figure
    """

    # --- Validate ---
    if not (isinstance(years, tuple) and len(years) == 2):
        raise ValueError("`years` must be a tuple like (2004, 2018).")

    start_year, end_year = years
    year_list = list(range(start_year, end_year + 1))

    # --- Prepare hourly demand ---
    demand.index = pd.to_datetime(demand.index)
    hourly = demand[[station]].resample("h").mean()

    # --- Collect ±30-day windows ---
    windows = []
    for yr in year_list:
        ref_date  = holiday_func(yr)
        win_start = ref_date - pd.Timedelta(days=30)
        win_end   = ref_date + pd.Timedelta(days=30)
        windows.append(hourly.loc[win_start:win_end].copy())

    combined = pd.concat(windows)
    combined["hour"] = combined.index.hour
    combined["date"] = combined.index.normalize()

    # --- Baseline: mean demand by hour, NON-holiday days only ---
    holiday_dates_set = {holiday_func(yr).normalize() for yr in year_list}
    non_holiday_mask  = ~combined["date"].isin(holiday_dates_set)
    baseline_by_hour  = combined[non_holiday_mask].groupby("hour")[station].mean()

    # --- Anomalies ---
    combined["anomaly"] = combined[station] - combined["hour"].map(baseline_by_hour)

    # --- Daily anomaly curves (all days) ---
    daily_curves = []
    for d, _ in combined.groupby(combined.index.date):
        day   = pd.date_range(pd.Timestamp(d), periods=24, freq="h")
        curve = combined["anomaly"].reindex(day).interpolate(limit_direction="both")
        curve.index = range(24)
        daily_curves.append(curve.rename(pd.Timestamp(d)))

    daily_matrix = pd.concat(daily_curves, axis=1)   # (24 x n_days)

    # --- Holiday-only curves (one per year) ---
    holiday_curves = []
    for yr in year_list:
        ref_date = holiday_func(yr)
        hours    = pd.date_range(ref_date, periods=24, freq="h")
        curve    = combined["anomaly"].reindex(hours).interpolate(limit_direction="both")
        curve.index = range(24)
        holiday_curves.append(curve.rename(yr))

    holiday_matrix = pd.concat(holiday_curves, axis=1)   # (24 x n_years)

    # --- Envelopes ---
    holiday_mean = holiday_matrix.mean(axis=1)
    hol_min      = holiday_matrix.min(axis=1)
    hol_max      = holiday_matrix.max(axis=1)

    non_hol_cols   = [c for c in daily_matrix.columns
                      if c.normalize() not in holiday_dates_set]
    non_hol_matrix = daily_matrix[non_hol_cols]
    bg_min         = non_hol_matrix.min(axis=1)
    bg_max         = non_hol_matrix.max(axis=1)

    # --- Per-hour one-sample t-test (H0: anomaly = 0) ---
    n        = holiday_matrix.shape[1]
    p_values = []
    for h in range(24):
        vals = holiday_matrix.loc[h].dropna()
        if len(vals) < 2:
            p_values.append(np.nan)
        else:
            _, p = stats.ttest_1samp(vals, popmean=0)
            p_values.append(p)
    p_values = np.array(p_values)

    # --- Overall effect size: eta² (holiday vs non-holiday ANOVA) ---
    holiday_flat     = holiday_matrix.values.flatten()
    holiday_flat     = holiday_flat[~np.isnan(holiday_flat)]
    non_holiday_flat = non_hol_matrix.values.flatten()
    non_holiday_flat = non_holiday_flat[~np.isnan(non_holiday_flat)]

    grand_mean  = np.concatenate([holiday_flat, non_holiday_flat]).mean()
    ss_between  = (
        len(holiday_flat)     * (holiday_flat.mean()     - grand_mean) ** 2
        + len(non_holiday_flat) * (non_holiday_flat.mean() - grand_mean) ** 2
    )
    ss_total        = np.sum(
        (np.concatenate([holiday_flat, non_holiday_flat]) - grand_mean) ** 2
    )
    eta_squared     = ss_between / ss_total if ss_total > 0 else np.nan
    f_stat, anova_p = stats.f_oneway(holiday_flat, non_holiday_flat)

    # --- Plot ---
    fig, (ax, ax_p) = plt.subplots(
        2, 1, figsize=(13, 8),
        gridspec_kw={"height_ratios": [3, 1]},
        sharex=True
    )

    hours = np.arange(24)

    # Grey band: min/max of all non-holiday days (background context)
    ax.fill_between(
        hours, bg_min, bg_max,
        color="lightgray", alpha=0.5,
        label="Non-holiday days (min/max envelope)"
    )

    # Salmon band: min/max envelope across holiday-year curves
    ax.fill_between(
        hours, hol_min, hol_max,
        color="salmon", alpha=0.5,
        label=f"{holiday_name} envelope (min/max, n={n} yrs)"
    )

    # Red line: mean holiday profile
    ax.plot(
        hours, holiday_mean,
        color="red", linewidth=2.5, marker="o",
        label=f"{holiday_name} Mean Anomaly"
    )

    ax.axhline(0, color="black", linewidth=1)
    ax.grid(axis="y", linestyle="-", linewidth=0.5, color="gray", alpha=0.3)
    ax.set_ylabel("Electricity Demand Anomaly (MW)")

    full_name = info.loc[station, "Name"]
    ax.set_title(
        f"{full_name} \u2014 {holiday_name} Anomaly ({start_year}\u2013{end_year}, \u00b130-Day Window)\n"
        f"\u03b7\u00b2 = {eta_squared:.4f}   |   ANOVA p = {anova_p:.4f}",
        fontsize=13
    )
    ax.legend(loc="upper left", fontsize=9)

    # Lower panel: per-hour -log10(p)
    bar_colors = ["red" if (not np.isnan(p) and p < alpha) else "steelblue"
                  for p in p_values]
    ax_p.bar(hours, -np.log10(p_values + 1e-10), color=bar_colors, alpha=0.8)
    ax_p.axhline(
        -np.log10(alpha), color="black", linestyle="--", linewidth=1,
        label=f"p = {alpha} threshold"
    )
    ax_p.set_ylabel("-log\u2081\u2080(p)")
    ax_p.set_xlabel("Hour of Day")
    ax_p.set_xticks(hours)
    ax_p.set_xticklabels([f"{h:02d}:00" for h in hours], rotation=45)
    ax_p.legend(fontsize=8)
    ax_p.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.4)

    # Print summary
    sig_hours = [h for h, p in enumerate(p_values) if not np.isnan(p) and p < alpha]
    print(f"\n{'─'*55}")
    print(f"  {holiday_name} | {station} | {start_year}\u2013{end_year}")
    print(f"{'─'*55}")
    print(f"  n years           : {n}")
    print(f"  \u03b7\u00b2 (effect size)  : {eta_squared:.4f}")
    print(f"  ANOVA F-stat      : {f_stat:.3f}")
    print(f"  ANOVA p-value     : {anova_p:.4f}")
    print(f"  Significant hours (p < {alpha}): {sig_hours}")
    print(f"{'─'*55}\n")

    fig.tight_layout()
    plt.show(fig)

# Single plot testing

In [ ]:
compare_holiday_ci(
    demand=demand,
    info=info,
    station="BLAKE",
    years=(2005, 2018),
    holiday_func=lambda y: pd.Timestamp(y, 12, 25),
    holiday_name="Christmas Day",
)